# Datos con huecos

**Sección 3.11 del libro**

En la vida real faltan datos. Dos estrategias: **rellenar** con el promedio o **eliminar** filas. Reúne los ejemplos `K3_iris-3` y `K3_iris-4` del libro.

> **Basado en:** Karatas, M. (2024). *Developing AI Applications: An Introduction*, cap. 3 «Artificial Neural Networks». Rheinwerk Publishing. Código original del autor, adaptado a Google Colab y comentado en español con fines educativos.


In [ ]:
import tensorflow as tf
import pandas as pd
from sklearn.model_selection import train_test_split

## Cargar los datos

Antes de ejecutar esta celda, sube **iris_empty.csv** al panel de archivos de Colab (ícono de carpeta → botón de subir). Si no lo tienes a mano, usa la línea comentada: lee el archivo directamente desde la página de la presentación.

In [ ]:
path = "iris_empty.csv"
# path = "https://sinergia-iesa.github.io/introduccion-a-la-creacion-de-redes-neuronales/data/iris_empty.csv"   # alternativa sin subir el archivo

data = pd.read_csv(path, delimiter=',')
print(data.head())
print("Columnas con huecos:", data.columns[data.isnull().any()])

## Estrategia A: rellenar

Números → promedio de la columna. Especie → la palabra `empty`, que se vuelve una **cuarta clase**: la salida necesita 4 neuronas.

In [ ]:
a = data.copy()
a['sepal.length'] = a['sepal.length'].fillna(a['sepal.length'].mean())
a['petal.length'] = a['petal.length'].fillna(a['petal.length'].mean())
a['species'] = a['species'].fillna('empty')
print('Huecos restantes:', a.columns[a.isnull().any()].tolist())

col = a['species'].astype('category').cat.codes
X = a.drop(['species'], axis=1)
train_data, test_data, train_col, test_col = train_test_split(X, col, test_size=0.2, random_state=42)
model = tf.keras.Sequential([
    tf.keras.Input(shape=(4,)),
    tf.keras.layers.Dense(32, activation='sigmoid'),
    tf.keras.layers.Dense(64, activation='sigmoid'),
    tf.keras.layers.Dense(4, activation='softmax')     # 3 especies + 'empty'
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_data, train_col, epochs=100, verbose=0)
print('A · rellenar → accuracy:', model.evaluate(test_data, test_col, verbose=0)[1])

## Estrategia B: eliminar filas incompletas

La salida vuelve a 3 neuronas.

In [ ]:
b = data.dropna()
print('Filas:', len(data), '→', len(b))

col = b['species'].astype('category').cat.codes
X = b.drop(['species'], axis=1)
train_data, test_data, train_col, test_col = train_test_split(X, col, test_size=0.2, random_state=42)
model = tf.keras.Sequential([
    tf.keras.Input(shape=(4,)),
    tf.keras.layers.Dense(32, activation='sigmoid'),
    tf.keras.layers.Dense(64, activation='sigmoid'),
    tf.keras.layers.Dense(3, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_data, train_col, epochs=100, verbose=0)
print('B · eliminar → accuracy:', model.evaluate(test_data, test_col, verbose=0)[1])

Prueba ambas y quédate con la que funcione mejor **para tus datos**.